# 10 · 전단과 비틀림 — KDS 14 20 22

| 항목 | 식 | 조문 |
|---|---|---|
| $V_c$ 간편식 | $\frac{1}{6}\lambda\sqrt{f_{ck}}b_wd$ | 식 (4.2-1) |
| $V_c$ 축압축 | $\frac{1}{6}(1+N_u/14A_g)\lambda\sqrt{f_{ck}}b_wd$ | 식 (4.2-2) |
| $V_c$ 정밀식 | $(0.16\lambda\sqrt{f_{ck}}+17.6\rho_wV_ud/M_u)b_wd$ | 식 (4.2-3) |
| $V_c$ 축인장 | $\frac{1}{6}(1+N_u/3.5A_g)\lambda\sqrt{f_{ck}}b_wd$ | 식 (4.2-6) |
| $V_s$ | $A_vf_{yt}d/s$ | 식 (4.3-3) |
| $A_{v,min}$ | $0.0625\sqrt{f_{ck}}b_ws/f_{yt} \ge 0.35b_ws/f_{yt}$ | 식 (4.3-1) |
| $T_{cr}$ | $\frac{1}{3}\lambda\sqrt{f_{ck}}A_{cp}^2/p_{cp}$ | 4.4.1 |

강도감소계수는 $\phi = 0.75$ 이다 (KDS 14 20 10 4.3.3(2)).

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np

# 한글 글꼴이 없는 환경에서도 그림이 깨지지 않도록 축 라벨은 ASCII 로 둔다
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 96

In [ ]:
from concreteproperties_kds.detailing import bar_area
from concreteproperties_kds.shear import (
    PHI_SHEAR,
    check_shear,
    check_torsion_section,
    concrete_shear_strength,
    cracking_torque,
    longitudinal_torsion_reinforcement,
    required_stirrup_spacing,
    torsion_negligible,
    torsional_strength,
)

FCK, FY = 27.0, 400.0
B_W, H, D, COVER = 400.0, 600.0, 550.0, 40.0
v_u = 320e3

a_v = 2 * bar_area("D13")   # D13 2가닥 스터럽

s_req = required_stirrup_spacing(
    v_u=v_u, fck=FCK, b_w=B_W, d=D, a_v=a_v, fyt=FY
)
s_use = 25.0 * int(s_req / 25.0)

print(f"필요 스터럽 간격 s = {s_req:.1f} mm  ->  배치 {s_use:.0f} mm")

In [ ]:
res = check_shear(v_u=v_u, fck=FCK, b_w=B_W, d=D, a_v=a_v, s=s_use, fyt=FY)
res.print_results()

## 축력이 $V_c$ 에 미치는 영향

압축은 $V_c$ 를 키우고 인장은 줄인다. KDS 는 압축과 인장에 서로 다른 식을
쓴다 — 식 (4.2-2) 와 식 (4.2-6).

In [ ]:
a_g = B_W * H
n_u = np.linspace(-1500e3, 3000e3, 300)
v_c = [
    concrete_shear_strength(
        fck=FCK, b_w=B_W, d=D, n_u=float(n), a_g=a_g
    ) / 1e3
    for n in n_u
]

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(n_u / 1e3, v_c)
ax.axvline(0, ls=":", color="grey", lw=0.8)
ax.axhline(
    concrete_shear_strength(fck=FCK, b_w=B_W, d=D) / 1e3,
    ls="--", color="grey", lw=0.8,
)
ax.set_xlabel("axial force, Nu (kN)   [+ compression]")
ax.set_ylabel("Vc (kN)")
ax.set_title("Effect of axial force on Vc")
ax.grid(alpha=0.3)
plt.show()

인장이 커지면 $V_c$ 가 0 에 도달한다. 그 뒤로는 전단철근이 전체 전단력을
받아야 한다 (KDS 14 20 22 4.2.1(1)③).

## 스터럽 간격에 따른 설계 전단강도

In [ ]:
spacings = np.arange(75, 401, 5.0)
phi_v_n = [
    check_shear(
        v_u=0, fck=FCK, b_w=B_W, d=D, a_v=a_v, s=float(s), fyt=FY
    ).phi_v_n / 1e3
    for s in spacings
]

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(spacings, phi_v_n)
ax.axhline(v_u / 1e3, ls="--", color="tab:red", label=f"Vu = {v_u / 1e3:.0f}")
ax.axvline(s_use, ls=":", color="tab:green", label=f"s = {s_use:.0f} mm")
ax.set_xlabel("stirrup spacing, s (mm)")
ax.set_ylabel("phi*Vn (kN)")
ax.set_title("Design shear strength vs stirrup spacing")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## 비틀림 (KDS 14 20 22 4.4, 4.5)

In [ ]:
a_cp, p_cp = B_W * H, 2 * (B_W + H)
a_oh = (B_W - 2 * COVER) * (H - 2 * COVER)
p_h = 2 * ((B_W - 2 * COVER) + (H - 2 * COVER))
t_u = 30e6

t_cr = cracking_torque(fck=FCK, a_cp=a_cp, p_cp=p_cp)
negligible = torsion_negligible(t_u=t_u, fck=FCK, a_cp=a_cp, p_cp=p_cp)

print(f"계수 비틀림모멘트  Tu           = {t_u / 1e6:8.2f} kN.m")
print(f"균열 비틀림모멘트  Tcr          = {t_cr / 1e6:8.2f} kN.m")
print(f"무시 한계      phi*Tcr/4        = "
      f"{PHI_SHEAR * t_cr / 4 / 1e6:8.2f} kN.m")
print(f"비틀림 무시 가능                = "
      f"{'예' if negligible else '아니오'}")

if not negligible:
    a_t = bar_area("D13")
    t_n = torsional_strength(a_t=a_t, s=s_use, a_oh=a_oh, fyt=FY)
    a_l = longitudinal_torsion_reinforcement(
        a_t=a_t, s=s_use, p_h=p_h, fyt=FY, fy=FY
    )
    print(f"공칭 비틀림강도    Tn           = {t_n / 1e6:8.2f} kN.m")
    print(f"설계 비틀림강도  phi*Tn         = "
          f"{PHI_SHEAR * t_n / 1e6:8.2f} kN.m")
    print(f"종방향 비틀림철근  Al           = {a_l:8.1f} mm^2")

demand, capacity, ok = check_torsion_section(
    v_u=v_u, t_u=t_u, fck=FCK, b_w=B_W, d=D, a_oh=a_oh, p_h=p_h
)
print(f"단면 크기  소요 {demand:.3f} <= 한계 {capacity:.3f} MPa"
      f"  {'만족' if ok else '불만족'}")